In [1]:
import pandas as pd

sheets = pd.read_excel("market_data.xlsx", sheet_name=None)
for name, d in sheets.items():
    print(f"Sheet: {name}, Shape: {d.shape}")

Sheet: source_A_industry_assoc, Shape: (37, 5)
Sheet: source_B_research_firm, Shape: (47, 5)
Sheet: drivers, Shape: (48, 4)
Sheet: fx_rates, Shape: (3, 2)


In [2]:
a = sheets["source_A_industry_assoc"]
b = sheets["source_B_research_firm"]
drivers = sheets["drivers"]
fx = sheets["fx_rates"]

In [3]:
a.head(10)

,country,year,market_value,currency,unit
0,Germany,2018,18.2,EUR,million
1,Germany,2019,19.7,EUR,million
2,Germany,2020,18.1,EUR,million
3,Germany,2021,23.6,EUR,million
4,Germany,2022,25.6,EUR,million
5,Germany,2023,28.6,EUR,million
6,Germany,2024,29.9,EUR,million
7,Germany,2025,34.6,EUR,million
8,France,2018,12.6,EUR,million
9,France,2019,13.3,EUR,million


In [4]:
a.dtypes

country             str
year              int64
market_value    float64
currency            str
unit                str
dtype: object

In [5]:
b.dtypes

country             str
year              int64
market_value    float64
currency            str
unit                str
dtype: object

In [6]:
a.groupby(['currency','unit']).size()

currency  unit   
EUR       million    37
dtype: int64

In [7]:
b.groupby(['currency','unit']).size()   

currency  unit   
DKK       million     8
EUR       billion    31
PLN       million     8
dtype: int64

In [8]:
a.groupby(["country","currency","unit"]).size()

country      currency  unit   
Austria      EUR       million    7
France       EUR       million    8
Germany      EUR       million    8
Netherlands  EUR       million    6
Poland       EUR       million    8
dtype: int64

In [9]:
b.groupby(["country","currency","unit"]).size()

country      currency  unit   
Austria      EUR       billion    8
Denmark      DKK       million    8
France       EUR       billion    8
Germany      EUR       billion    7
Netherlands  EUR       billion    8
Poland       PLN       million    8
dtype: int64

In [10]:
#coverage
a.groupby("country")["year"].agg(["count","min","max"])

,count,min,max
country,,,
Austria,7,2018,2025
France,8,2018,2025
Germany,8,2018,2025
Netherlands,6,2018,2025
Poland,8,2018,2025


In [11]:
b.groupby("country")["year"].agg(["count","min","max"])

,count,min,max
country,,,
Austria,8,2018,2025
Denmark,8,2018,2025
France,8,2018,2025
Germany,7,2018,2025
Netherlands,8,2018,2025
Poland,8,2018,2025


In [12]:
a[a["country"] == "France"]

,country,year,market_value,currency,unit
8,France,2018,12.6,EUR,million
9,France,2019,13.3,EUR,million
10,France,2020,12.5,EUR,million
11,France,2021,16.3,EUR,million
12,France,2022,176.0,EUR,million
13,France,2023,20.2,EUR,million
14,France,2024,21.4,EUR,million
15,France,2025,23.6,EUR,million


In [13]:
b[b["country"] == "Austria"]

,country,year,market_value,currency,unit
31,Austria,2018,0.002,EUR,billion
32,Austria,2019,0.002,EUR,billion
33,Austria,2020,0.002,EUR,billion
34,Austria,2021,0.003,EUR,billion
35,Austria,2022,0.003,EUR,billion
36,Austria,2023,0.003,EUR,billion
37,Austria,2024,0.003,EUR,billion
38,Austria,2025,0.004,EUR,billion


Standardise units and currency

In [14]:
fx

,currency,eur_per_unit
0,EUR,1.000000
1,PLN,0.232558
2,DKK,0.134228


In [15]:
UNIT_FACTOR = {"million": 1, "billion": 1000}

def standarsize(df):
    out = df.merge(fx, on="currency", how="left")
    out["unit_factor"] = out["unit"].map(UNIT_FACTOR)
    out["value_eur_m"] = out["market_value"] * out["unit_factor"] * out["eur_per_unit"]
    return out

a_std = standarsize(a)
b_std = standarsize(b)

In [16]:
b_std.head(10)

,country,year,market_value,currency,unit,eur_per_unit,unit_factor,value_eur_m
0,Germany,2018,0.021,EUR,billion,1.0,1000,21.0
1,Germany,2020,0.021,EUR,billion,1.0,1000,21.0
2,Germany,2021,0.029,EUR,billion,1.0,1000,29.0
3,Germany,2022,0.031,EUR,billion,1.0,1000,31.0
4,Germany,2023,0.033,EUR,billion,1.0,1000,33.0
5,Germany,2024,0.036,EUR,billion,1.0,1000,36.0
6,Germany,2025,0.041,EUR,billion,1.0,1000,41.0
7,France,2018,0.014,EUR,billion,1.0,1000,14.0
8,France,2019,0.016,EUR,billion,1.0,1000,16.0
9,France,2020,0.015,EUR,billion,1.0,1000,15.0


In [17]:
a_std["value_eur_m"].isna().sum()

np.int64(0)

In [18]:
b_std["value_eur_m"].isna().sum()

np.int64(0)

In [19]:
#compare and join
compare = a_std[["country", "year", "value_eur_m"]].merge(
    b_std[["country", "year", "value_eur_m"]],
    on=["country", "year"],
    suffixes=("_a", "_b"),
)
compare["ratio_b_a"] = compare["value_eur_m_b"] / compare["value_eur_m_a"]
compare.round(2)

,country,year,value_eur_m_a,value_eur_m_b,ratio_b_a
0,Germany,2018,18.2,21.00,1.15
1,Germany,2020,18.1,21.00,1.16
2,Germany,2021,23.6,29.00,1.23
3,Germany,2022,25.6,31.00,1.21
4,Germany,2023,28.6,33.00,1.15
5,Germany,2024,29.9,36.00,1.20
6,Germany,2025,34.6,41.00,1.18
7,France,2018,12.6,14.00,1.11
8,France,2019,13.3,16.00,1.20
9,France,2020,12.5,15.00,1.20


In [20]:
#quality checks
countries = sorted(set(a_std["country"]) | set(b_std["country"]))
years = list(range(2018, 2026))

grid = pd.MultiIndex.from_product(
    [countries, years], names=["country", "year"]
).to_frame(index=False)

In [21]:
#completeness check
qa = grid.merge(a_std[["country", "year", "value_eur_m"]],
                on=["country", "year"],
                how="left")
qa["flag_missing"] = qa["value_eur_m"].isna()
qa["flag_missing"].sum()

np.int64(11)

In [22]:
#plausibility check (YoY growth)
qa = qa.sort_values(["country", "year"])
qa["yoy"] = qa.groupby("country")["value_eur_m"].pct_change(fill_method=None)
qa["flag_yoy_jump"] = qa["yoy"].abs() > 0.5

In [ ]:
#consistency check (source A vs source B)
qa = qa.merge(b_std[["country", "year", "value_eur_m"]].rename(columns={"value_eur_m": "value_b"}),
              on=["country", "year"], how="left")
qa["ratio_b_to_a"] = qa["value_b"] / qa["value_eur_m"]

typical_ratio = qa["ratio_b_to_a"].median()
qa["flag_source_mismatch"] = (qa["ratio_b_to_a"] / typical_ratio - 1).abs() > 0.5

print("Typical ratio:", round(typical_ratio, 2))
qa[qa["flag_source_mismatch"]]

Typical ratio: 1.19


,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
20,France,2022,176.0,False,9.797546,True,22.0,0.125,True


In [24]:
qa

,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
0,Austria,2018,2.0,False,NaN,False,2.000000,1.000000,False
1,Austria,2019,2.2,False,0.100000,False,2.000000,0.909091,False
2,Austria,2020,1.9,False,-0.136364,False,2.000000,1.052632,False
3,Austria,2021,2.5,False,0.315789,False,3.000000,1.200000,False
4,Austria,2022,2.6,False,0.040000,False,3.000000,1.153846,False
5,Austria,2023,NaN,True,NaN,False,3.000000,NaN,False
6,Austria,2024,3.0,False,NaN,False,3.000000,1.000000,False
7,Austria,2025,3.3,False,0.100000,False,4.000000,1.212121,False
8,Denmark,2018,NaN,True,NaN,False,1.879195,NaN,False
9,Denmark,2019,NaN,True,NaN,False,2.013423,NaN,False


In [25]:
flag_cols = ["flag_missing", "flag_yoy_jump", "flag_source_mismatch"]
qa[flag_cols].sum()

flag_missing            11
flag_yoy_jump            2
flag_source_mismatch     1
dtype: int64

In [28]:
qa[qa[flag_cols].any(axis=1)]

,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
5,Austria,2023,NaN,True,NaN,False,3.000000,NaN,False
8,Denmark,2018,NaN,True,NaN,False,1.879195,NaN,False
9,Denmark,2019,NaN,True,NaN,False,2.013423,NaN,False
10,Denmark,2020,NaN,True,NaN,False,2.013423,NaN,False
11,Denmark,2021,NaN,True,NaN,False,2.281879,NaN,False
12,Denmark,2022,NaN,True,NaN,False,2.684564,NaN,False
13,Denmark,2023,NaN,True,NaN,False,2.953020,NaN,False
14,Denmark,2024,NaN,True,NaN,False,2.953020,NaN,False
15,Denmark,2025,NaN,True,NaN,False,3.624161,NaN,False
20,France,2022,176.0,False,9.797546,True,22.000000,0.125000,True


In [ ]:
#Reconcile two sources
